# BreastHistCNN (BH-CNN)
## CNN, Swin Transformer y aprendizaje Multi-task para clasificación histopatológica de mama

**Guillermo Quijije · Paúl Calderón**

Este notebook reúne la metodología, las decisiones experimentales y la reproducción de los resultados finales del proyecto BH-CNN. La tarea principal es clasificar imágenes histopatológicas de tejido mamario en **Benigno** y **Maligno**, utilizando al paciente como unidad de agrupación.

La ejecución normal **no vuelve a entrenar ni retunar los modelos**. Las secciones de entrenamiento, selección de hiperparámetros y fine-tuning documentan el procedimiento utilizado durante el proyecto. La parte ejecutable reconstruye los tres predictores congelados desde sus checkpoints y reproduce TEST.

### Archivos requeridos

Ubicar en Google Drive, preferentemente en `MyDrive/Colab Notebooks`:

- `BreaKHis_v1.tar.gz`
- `split_manifest.csv`
- `BH_CNN_CHECKPOINTS_FINALES_9PT.zip`

### Predictores finales

| Modelo | Seeds | Threshold | Parámetros |
|---|---|---:|---:|
| CNN Shallow | 20261007, 20261008, 20261009 | 0.17 | 93.601 |
| Swin-T | 20261007, 20261008, 20261009 | 0.47 | 27.520.123 |
| CNN Multi-task | 20261007, 20261008, 20261009 | 0.05 | 94.633 |

Para Multi-task se utiliza `λ = 0.10`. Los tres predictores son ensembles construidos mediante la media aritmética de probabilidades.

## 1. Problema y objetivos

BreaKHis contiene múltiples imágenes por paciente y distintas magnificaciones del tejido. Un split aleatorio por imagen puede dejar muestras del mismo paciente en TRAIN y TEST, introduciendo fuga de información. Por esta razón, el proyecto utiliza una partición estricta por paciente y mantiene el mismo TEST para las tres familias de modelos.

El objetivo general fue desarrollar, comparar y evaluar estrategias de Deep Learning para clasificación binaria Benigno/Maligno, considerando no solo ROC-AUC, sino también sensibilidad, especificidad, F1, falsos negativos, costo computacional y reproducibilidad.

Los objetivos específicos fueron:

1. Auditar y depurar BreaKHis y congelar un split sin solapamiento de pacientes.
2. Comparar tres CNN propias entrenadas desde cero.
3. Evaluar hiperparámetros, regularización, augmentation, desbalance y threshold utilizando VALIDATION.
4. Evaluar un Transformer mediante Transfer Learning y fine-tuning.
5. Evaluar una CNN Multi-task con subtipo histológico como tarea auxiliar.
6. Comparar los predictores finales sobre el mismo TEST y aplicar Grad-CAM como atribución de decisión.

BH-CNN es un prototipo académico. No constituye un dispositivo médico ni sustituye el criterio del patólogo.

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

from pathlib import Path
import gc
import hashlib
import json
import shutil
import tarfile
import zipfile

import numpy as np
import pandas as pd
from PIL import Image

import matplotlib.pyplot as plt

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torchvision.models import swin_t

from sklearn.metrics import (
    accuracy_score,
    confusion_matrix,
    f1_score,
    precision_score,
    roc_auc_score,
    roc_curve
)

from IPython.display import display

MYDRIVE = Path("/content/drive/MyDrive")
PREFERRED = MYDRIVE / "Colab Notebooks"
WORK = Path("/content/BH_CNN_MEDIUM")
CHECKPOINT_DIR = WORK / "checkpoints"
TEST_DIR = WORK / "test_images"
RESULTS_DIR = PREFERRED / "BH_CNN_MEDIUM_RESULTS"
FIG_DIR = RESULTS_DIR / "figures"

for p in [WORK, CHECKPOINT_DIR, TEST_DIR, RESULTS_DIR, FIG_DIR]:
    p.mkdir(parents=True, exist_ok=True)

def resolve_file(filename):
    preferred = PREFERRED / filename
    if preferred.exists():
        return preferred
    matches = list(MYDRIVE.rglob(filename))
    if len(matches) == 1:
        return matches[0]
    if len(matches) == 0:
        raise FileNotFoundError(filename)
    raise RuntimeError(f"Se encontraron varias copias de {filename}: {matches}")

DATASET_TGZ = resolve_file("BreaKHis_v1.tar.gz")
MANIFEST_PATH = resolve_file("split_manifest.csv")
CHECKPOINT_ZIP = resolve_file("BH_CNN_CHECKPOINTS_FINALES_9PT.zip")

SEEDS = [20261007, 20261008, 20261009]
THRESHOLDS = {"CNN": 0.17, "Swin-T": 0.47, "Multi-task": 0.05}
MULTITASK_LAMBDA = 0.10

TARGET_H = 224
TARGET_W = 320
BATCH_SIZE = 32

CNN_MEAN = np.array([0.7910038829, 0.6160358787, 0.7618963122], dtype=np.float32)
CNN_STD = np.array([0.1280490756, 0.1800721437, 0.1118019968], dtype=np.float32)
IMAGENET_MEAN = np.array([0.485, 0.456, 0.406], dtype=np.float32)
IMAGENET_STD = np.array([0.229, 0.224, 0.225], dtype=np.float32)

MANIFEST_SHA = "e706c0a92df5b9bc3d1de94c9d541ee80077ef79a291c4f1460d9285f27ee1ba"

CHECKPOINTS = {
    "CNN": {
        20261007: ("EXP001_seed_20261007.pt", "b9a724e021de619c36f0dbd1b9ea38d66f6f7b3a9c04a8146b605dcf1cba36e0"),
        20261008: ("EXP001_seed_20261008.pt", "65baca25fffc6941f58fc5cba851692c514aee38155b1c8038ab8e64c2fc8dc7"),
        20261009: ("EXP001_seed_20261009.pt", "7d6c09cbe08526a7c5e6a4f5d72002a4ddc0198ac805756ac3517a9337a979d7")
    },
    "Swin-T": {
        20261007: ("best_SWIN_T_S20261007_STAGE72.pt", "eea672c56446d1536224b8e7ba9519d9a51f5d363d18c644f12495e1bb14cdb0"),
        20261008: ("best_SWIN_T_S20261008_STAGE72.pt", "ab7aed6e5c892173e4853a33525e9dea1e84ac9bd550f15f14ae9082d6af0287"),
        20261009: ("best_SWIN_T_S20261009_STAGE72.pt", "66325bb9c7dc08e6538bbd7583016b5d9da2fd31a5fe8d2ee64aa052a7ae5960")
    },
    "Multi-task": {
        20261007: ("best_MULTITASK_L0p10_S20261007.pt", "f14cae96d7fd8c59371ea9b919c29804cc32e8a24549e9599492755532711f61"),
        20261008: ("best_MULTITASK_L0p10_S20261008.pt", "2c4ef77cb1e83640875e42fe9ef31dd44b1e1c1c7366cf7755bfbc4d77169a77"),
        20261009: ("best_MULTITASK_L0p10_S20261009.pt", "896d33ef1f40c74b50fc6faa081a3388888923c42f60cc9180f8dbb1eecbd54d")
    }
}

EXPECTED = {
    "CNN": {"auc": 0.959293, "tn": 256, "fp": 155, "fn": 28, "tp": 907},
    "Swin-T": {"auc": 0.983156, "tn": 388, "fp": 23, "fn": 81, "tp": 854},
    "Multi-task": {"auc": 0.969203, "tn": 255, "fp": 156, "fn": 12, "tp": 923}
}

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Dispositivo:", DEVICE)
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

## 2. Dataset BreaKHis

El dataset original contenía **7.909 imágenes de 82 pacientes**. La auditoría eliminó **125 duplicados exactos a nivel de píxeles**, dejando un universo operativo de **7.784 imágenes y 81 pacientes**, sin archivos corruptos ni duplicados exactos restantes.

La distribución final es:

- 2.480 imágenes benignas.
- 5.304 imágenes malignas.
- 24 pacientes benignos.
- 57 pacientes malignos.
- 8 subtipos histológicos.
- Magnificaciones 40X, 100X, 200X y 400X.
- 7.710 imágenes de 700×460 y 74 de 700×456.

Subtipos benignos: A, F, PT y TA.  
Subtipos malignos: DC, LC, MC y PC.

La variable objetivo es `Benigno=0 / Maligno=1`.

In [ ]:
def sha256_file(path, chunk_size=1024 * 1024):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for chunk in iter(lambda: f.read(chunk_size), b""):
            h.update(chunk)
    return h.hexdigest()

if sha256_file(MANIFEST_PATH) != MANIFEST_SHA:
    raise RuntimeError("El manifest no coincide con el split congelado del proyecto.")

with zipfile.ZipFile(CHECKPOINT_ZIP, "r") as zf:
    expected_names = {v[0] for family in CHECKPOINTS.values() for v in family.values()}
    observed_names = set(zf.namelist())
    if observed_names != expected_names:
        raise RuntimeError("El ZIP de checkpoints no contiene exactamente los 9 modelos oficiales.")

    for family, seeds in CHECKPOINTS.items():
        for seed, (name, expected_sha) in seeds.items():
            target = CHECKPOINT_DIR / name
            with zf.open(name) as src, open(target, "wb") as dst:
                shutil.copyfileobj(src, dst)
            if sha256_file(target) != expected_sha:
                raise RuntimeError(f"SHA-256 incorrecto: {name}")

print("Manifest y 9 checkpoints verificados.")

In [ ]:
def first_column(df, candidates):
    lookup = {str(c).lower(): c for c in df.columns}
    for name in candidates:
        if name.lower() in lookup:
            return lookup[name.lower()]
    return None

manifest = pd.read_csv(MANIFEST_PATH)

split_col = first_column(manifest, ["split", "partition", "set"])
patient_col = first_column(manifest, ["patient_id", "patient", "patientid"])
subtype_col = first_column(manifest, ["subtype", "tumor_subtype", "histological_subtype"])
path_col = first_column(manifest, ["relative_path", "filepath", "file_path", "path", "localizer", "filename", "file"])

if None in [split_col, patient_col, subtype_col, path_col]:
    raise RuntimeError("El manifest no contiene las columnas mínimas requeridas.")

manifest["subtype_clean"] = manifest[subtype_col].astype(str).str.strip().str.upper()
manifest["y_true"] = manifest["subtype_clean"].isin({"DC", "LC", "MC", "PC"}).astype(int)
manifest["sample_key"] = (
    manifest[path_col].astype(str)
    .str.replace("\\", "/", regex=False)
    .map(lambda x: Path(x).name)
)
manifest["split_clean"] = manifest[split_col].astype(str).str.strip().str.lower()

split_summary = (
    manifest.groupby("split_clean")
    .agg(
        Pacientes=(patient_col, lambda x: x.astype(str).nunique()),
        Imagenes=("sample_key", "size"),
        Benignas=("y_true", lambda x: int((x == 0).sum())),
        Malignas=("y_true", lambda x: int((x == 1).sum()))
    )
    .reindex(["train", "validation", "test"])
)

display(split_summary)

expected_split = {
    "train": (53, 5101),
    "validation": (14, 1337),
    "test": (14, 1346)
}

for split_name, (n_patients, n_images) in expected_split.items():
    row = split_summary.loc[split_name]
    if int(row["Pacientes"]) != n_patients or int(row["Imagenes"]) != n_images:
        raise RuntimeError(f"Conteos inesperados en {split_name}.")

patients_by_split = {
    s: set(manifest.loc[manifest["split_clean"] == s, patient_col].astype(str))
    for s in ["train", "validation", "test"]
}

if patients_by_split["train"] & patients_by_split["validation"]:
    raise RuntimeError("Leakage TRAIN/VALIDATION.")
if patients_by_split["train"] & patients_by_split["test"]:
    raise RuntimeError("Leakage TRAIN/TEST.")
if patients_by_split["validation"] & patients_by_split["test"]:
    raise RuntimeError("Leakage VALIDATION/TEST.")

class_counts = manifest["y_true"].value_counts().reindex([0, 1]).rename(index={0: "Benigno", 1: "Maligno"})
subtype_order = ["A", "F", "PT", "TA", "DC", "LC", "MC", "PC"]
subtype_counts = manifest["subtype_clean"].value_counts().reindex(subtype_order)

fig, ax = plt.subplots(figsize=(6, 4))
class_counts.plot(kind="bar", ax=ax)
ax.set_title("Distribución final de la tarea binaria")
ax.set_xlabel("")
ax.set_ylabel("Imágenes")
ax.tick_params(axis="x", rotation=0)
plt.tight_layout()
fig.savefig(FIG_DIR / "distribucion_binaria.png", dpi=180, bbox_inches="tight")
plt.show()

fig, ax = plt.subplots(figsize=(8, 4))
subtype_counts.plot(kind="bar", ax=ax)
ax.set_title("Distribución por subtipo histológico")
ax.set_xlabel("Subtipo")
ax.set_ylabel("Imágenes")
ax.tick_params(axis="x", rotation=0)
plt.tight_layout()
fig.savefig(FIG_DIR / "distribucion_subtipos.png", dpi=180, bbox_inches="tight")
plt.show()

## 3. Split por paciente

La imagen es la unidad de entrenamiento, pero el **paciente es la unidad de agrupación**. Todas las imágenes de un paciente permanecen en un único conjunto.

Se generaron 100.000 candidatos de partición. La selección final fue **Rank 23 / Candidate 61147**, porque TRAIN y TEST contienen los ocho subtipos y VALIDATION conserva los cuatro subtipos malignos. El split se congeló y se reutilizó en CNN, Swin-T y Multi-task.

TRAIN se utiliza para actualizar pesos. VALIDATION se utiliza para arquitectura, early stopping, hiperparámetros y threshold. TEST se reserva para la evaluación final.

## 4. Preprocesamiento

Para CNN y Multi-task se utiliza una entrada de **320×224**. El procedimiento conserva la relación de aspecto mediante resize proporcional y padding simétrico.

La media y desviación estándar se calcularon únicamente con TRAIN:

\[
\mu=(0.7910039,\ 0.6160359,\ 0.7618963)
\]

\[
\sigma=(0.1280491,\ 0.1800721,\ 0.1118020)
\]

El padding se rellena con la media RGB de TRAIN para que, después de normalizar, esa región quede aproximadamente en cero. Swin-T usa la misma geometría, pero normalización de ImageNet por partir de pesos preentrenados.

In [ ]:
def resize_pad_normalize(img, mean, std):
    img = img.convert("RGB")
    w, h = img.size
    scale = min(TARGET_W / w, TARGET_H / h)
    new_w = max(1, int(round(w * scale)))
    new_h = max(1, int(round(h * scale)))
    img = img.resize((new_w, new_h), Image.Resampling.BILINEAR)

    arr = np.asarray(img, dtype=np.float32) / 255.0
    canvas = np.empty((TARGET_H, TARGET_W, 3), dtype=np.float32)
    canvas[...] = mean.reshape(1, 1, 3)

    x0 = (TARGET_W - new_w) // 2
    y0 = (TARGET_H - new_h) // 2
    canvas[y0:y0 + new_h, x0:x0 + new_w, :] = arr

    canvas = (canvas - mean.reshape(1, 1, 3)) / std.reshape(1, 1, 3)
    return torch.from_numpy(canvas).permute(2, 0, 1).contiguous()

def resize_pad_display(img):
    img = img.convert("RGB")
    w, h = img.size
    scale = min(TARGET_W / w, TARGET_H / h)
    new_w = max(1, int(round(w * scale)))
    new_h = max(1, int(round(h * scale)))
    img = img.resize((new_w, new_h), Image.Resampling.BILINEAR)

    arr = np.asarray(img, dtype=np.float32) / 255.0
    canvas = np.empty((TARGET_H, TARGET_W, 3), dtype=np.float32)
    canvas[...] = CNN_MEAN.reshape(1, 1, 3)

    x0 = (TARGET_W - new_w) // 2
    y0 = (TARGET_H - new_h) // 2
    canvas[y0:y0 + new_h, x0:x0 + new_w, :] = arr
    return canvas

## 5. Selección de la CNN

Se compararon tres arquitecturas propias entrenadas desde cero y bajo las mismas condiciones:

- **CNN Shallow:** 32 → 64 → 128, una convolución por bloque.
- **Deep/VGG-like:** 32 → 64 → 128 → 256, dos convoluciones por bloque.
- **Residual:** bloques residuales propios y proyecciones 1×1.

La métrica primaria fue ROC-AUC de VALIDATION. En la primera seed, Shallow obtuvo 0.966055, Deep 0.955526 y Residual 0.941661. Debido a la proximidad entre las dos primeras, Shallow y Deep se repitieron con tres seeds.

Después de las repeticiones, Shallow alcanzó AUC medio 0.965050 y Deep 0.954051. La diferencia media fue 0.010999. Shallow fue seleccionada por rendimiento y parsimonia.

In [ ]:
architecture_results = pd.DataFrame([
    {"Arquitectura": "CNN Shallow", "Parametros": 93601, "AUC_seed_20261007": 0.966055, "AUC_medio_3seeds": 0.965050, "SD": 0.004078},
    {"Arquitectura": "Deep/VGG-like", "Parametros": 1173473, "AUC_seed_20261007": 0.955526, "AUC_medio_3seeds": 0.954051, "SD": 0.005279},
    {"Arquitectura": "Residual", "Parametros": 1226657, "AUC_seed_20261007": 0.941661, "AUC_medio_3seeds": np.nan, "SD": np.nan}
])

display(architecture_results)

fig, ax = plt.subplots(figsize=(7, 4))
architecture_results.plot(x="Arquitectura", y="AUC_seed_20261007", kind="bar", legend=False, ax=ax)
ax.set_ylim(0.90, 0.98)
ax.set_ylabel("ROC-AUC VALIDATION")
ax.set_xlabel("")
ax.set_title("Screening arquitectónico inicial")
ax.tick_params(axis="x", rotation=0)
plt.tight_layout()
fig.savefig(FIG_DIR / "screening_arquitecturas.png", dpi=180, bbox_inches="tight")
plt.show()

### Arquitectura seleccionada

```python
class CNNShallow(nn.Module):
    def __init__(self, dropout=0.30):
        super().__init__()
        self.features = nn.Sequential(
            ConvBNReLU(3, 32), nn.MaxPool2d(2),
            ConvBNReLU(32, 64), nn.MaxPool2d(2),
            ConvBNReLU(64, 128), nn.MaxPool2d(2)
        )
        self.gap = nn.AdaptiveAvgPool2d((1, 1))
        self.dropout = nn.Dropout(dropout)
        self.classifier = nn.Linear(128, 1)

    def forward(self, x):
        x = self.features(x)
        x = self.gap(x)
        x = torch.flatten(x, 1)
        x = self.dropout(x)
        return self.classifier(x).squeeze(1)
```

## 6. Entrenamiento y validación de la CNN

La configuración base utilizó `BCEWithLogitsLoss` sin ponderar, Adam, `lr=0.001`, `dropout=0.30`, `weight_decay=0`, batch 32, sin scheduler, balance natural y sin augmentation. El máximo fue 50 épocas, con early stopping `patience=7` y `min_delta=0.0001`, monitorizando ROC-AUC de VALIDATION.

El checkpoint se guardó cuando:

\[
AUC_{nuevo} > AUC_{mejor} + 10^{-4}
\]

Las siguientes curvas corresponden a la seed oficial `20261007`. Se incluyen como evidencia histórica del entrenamiento; esta sección no reentrena el modelo.

In [ ]:
cnn_history = pd.DataFrame([{'epoch': 1, 'train_loss': 0.4130569191463693, 'val_loss': 0.4644209730955051, 'val_roc_auc': 0.9208372962949188}, {'epoch': 2, 'train_loss': 0.348223643264591, 'val_loss': 0.4091517573519259, 'val_roc_auc': 0.9469954994925636}, {'epoch': 3, 'train_loss': 0.3171534416746144, 'val_loss': 0.5417271380827511, 'val_roc_auc': 0.9417962710081242}, {'epoch': 4, 'train_loss': 0.2874226589088135, 'val_loss': 0.5631763369803475, 'val_roc_auc': 0.936546564577234}, {'epoch': 5, 'train_loss': 0.2755758676417223, 'val_loss': 0.6269087183359584, 'val_roc_auc': 0.9381671723317092}, {'epoch': 6, 'train_loss': 0.2435150152765005, 'val_loss': 0.6977072672426656, 'val_roc_auc': 0.9159276518190658}, {'epoch': 7, 'train_loss': 0.2336985609433716, 'val_loss': 0.3998504321166803, 'val_roc_auc': 0.924660336555066}, {'epoch': 8, 'train_loss': 0.2205603283670597, 'val_loss': 0.3376848962027008, 'val_roc_auc': 0.9524789985175424}, {'epoch': 9, 'train_loss': 0.2205922947405974, 'val_loss': 0.5995668798751945, 'val_roc_auc': 0.9501357591086126}, {'epoch': 10, 'train_loss': 0.2209105457562111, 'val_loss': 0.581104277650545, 'val_roc_auc': 0.9387861913592382}, {'epoch': 11, 'train_loss': 0.2074216335772346, 'val_loss': 0.4884426911196426, 'val_roc_auc': 0.9440412112581762}, {'epoch': 12, 'train_loss': 0.1972894282276502, 'val_loss': 0.7212244918654904, 'val_roc_auc': 0.9402208277320524}, {'epoch': 13, 'train_loss': 0.1971811253933129, 'val_loss': 0.5896808460031594, 'val_roc_auc': 0.9181938459413076}, {'epoch': 14, 'train_loss': 0.1951763432263346, 'val_loss': 0.4508794252785088, 'val_roc_auc': 0.960393409174234}, {'epoch': 15, 'train_loss': 0.1901672014101559, 'val_loss': 0.6525304876964486, 'val_roc_auc': 0.944787753518844}, {'epoch': 16, 'train_loss': 0.1804256107942236, 'val_loss': 0.5285657279600202, 'val_roc_auc': 0.948214940409456}, {'epoch': 17, 'train_loss': 0.1773640558333916, 'val_loss': 0.4526694439178808, 'val_roc_auc': 0.9640198511166252}, {'epoch': 18, 'train_loss': 0.1875519761328188, 'val_loss': 0.8227875151559321, 'val_roc_auc': 0.9365226539710204}, {'epoch': 19, 'train_loss': 0.1730904493395475, 'val_loss': 0.4815939918728845, 'val_roc_auc': 0.9235843592754556}, {'epoch': 20, 'train_loss': 0.1777896517820158, 'val_loss': 0.3322792302151179, 'val_roc_auc': 0.9275110121625284}, {'epoch': 21, 'train_loss': 0.179772288953877, 'val_loss': 0.4072452755387184, 'val_roc_auc': 0.9660549093788025}, {'epoch': 22, 'train_loss': 0.1691096504469919, 'val_loss': 0.4278174766249503, 'val_roc_auc': 0.9475029356910962}, {'epoch': 23, 'train_loss': 0.1629527903285547, 'val_loss': 0.3160569986458046, 'val_roc_auc': 0.9336693216295344}, {'epoch': 24, 'train_loss': 0.1544164474443743, 'val_loss': 0.6878882720230584, 'val_roc_auc': 0.94732759124553}, {'epoch': 25, 'train_loss': 0.164896578832342, 'val_loss': 0.7992711034655661, 'val_roc_auc': 0.8847030568381677}, {'epoch': 26, 'train_loss': 0.1532752690210549, 'val_loss': 0.9391012920669948, 'val_roc_auc': 0.9580820505735888}, {'epoch': 27, 'train_loss': 0.155566326102435, 'val_loss': 0.773166908911118, 'val_roc_auc': 0.9407149802604662}, {'epoch': 28, 'train_loss': 0.1566415206985458, 'val_loss': 0.5362068479776204, 'val_roc_auc': 0.9437064627711862}])

fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(cnn_history["epoch"], cnn_history["train_loss"], label="Train loss")
ax.plot(cnn_history["epoch"], cnn_history["val_loss"], label="Validation loss")
ax.set_xlabel("Época")
ax.set_ylabel("Loss")
ax.set_title("CNN Shallow · seed 20261007")
ax.legend()
plt.tight_layout()
fig.savefig(FIG_DIR / "cnn_training_loss_historica.png", dpi=180, bbox_inches="tight")
plt.show()

fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(cnn_history["epoch"], cnn_history["val_roc_auc"])
ax.set_xlabel("Época")
ax.set_ylabel("ROC-AUC")
ax.set_title("CNN Shallow · ROC-AUC de VALIDATION")
plt.tight_layout()
fig.savefig(FIG_DIR / "cnn_validation_auc_historica.png", dpi=180, bbox_inches="tight")
plt.show()

El entrenamiento original siguió este patrón general:

```python
for epoch in range(max_epochs):
    model.train()
    for images, labels in train_loader:
        optimizer.zero_grad()
        logits = model(images)
        loss = criterion(logits, labels.float())
        loss.backward()
        optimizer.step()

    model.eval()
    val_probability = infer_validation(model, validation_loader)
    val_auc = roc_auc_score(y_validation, val_probability)

    if val_auc > best_auc + min_delta:
        best_auc = val_auc
        save_checkpoint(model)
        patience_counter = 0
    else:
        patience_counter += 1

    if patience_counter >= patience:
        break
```

Este código se muestra como referencia metodológica. El notebook final utiliza directamente los checkpoints ya congelados.

## 7. Selección de hiperparámetros, augmentation y desbalance

La optimización fue secuencial. Cada factor se comparó contra el baseline retenido y se utilizó un umbral práctico de `ΔAUC = 0.01` para justificar un cambio.

Ninguna de las alternativas finales alcanzó una mejora material de 0.01. Por ello se mantuvo la configuración más simple.

En augmentation se evaluó una configuración con flips y rotación ±5°. La mejora descriptiva fue `ΔAUC=+0.00382`, insuficiente para reemplazar A0. En desbalance se compararon distribución natural, BCE ponderada, sampling por clase y sampling jerárquico; se mantuvo la distribución natural.

In [ ]:
tuning = pd.DataFrame([
    {"Factor": "Learning rate", "Baseline": "0.001", "Alternativa": "0.0003", "Delta_AUC": 0.00476},
    {"Factor": "Weight decay", "Baseline": "0", "Alternativa": "1e-3", "Delta_AUC": 0.00634},
    {"Factor": "Dropout", "Baseline": "0.30", "Alternativa": "0.50", "Delta_AUC": 0.00068},
    {"Factor": "Optimizer", "Baseline": "Adam", "Alternativa": "SGD + momentum", "Delta_AUC": -0.00119},
    {"Factor": "Batch size", "Baseline": "32", "Alternativa": "16", "Delta_AUC": -0.00028},
    {"Factor": "Scheduler", "Baseline": "None", "Alternativa": "ReduceLROnPlateau", "Delta_AUC": -0.00019},
    {"Factor": "Augmentation", "Baseline": "A0", "Alternativa": "A1", "Delta_AUC": 0.00382}
])

display(tuning)

fig, ax = plt.subplots(figsize=(8, 4.5))
tuning.plot(x="Factor", y="Delta_AUC", kind="barh", legend=False, ax=ax)
ax.axvline(0.01, linestyle="--", linewidth=1)
ax.set_xlabel("Δ ROC-AUC frente al baseline")
ax.set_ylabel("")
ax.set_title("Selección de hiperparámetros")
plt.tight_layout()
fig.savefig(FIG_DIR / "tuning_delta_auc.png", dpi=180, bbox_inches="tight")
plt.show()

## 8. Threshold de clasificación

El threshold no se fijó automáticamente en 0.50. Se seleccionó exclusivamente con VALIDATION utilizando una regla predeclarada:

1. maximizar sensibilidad maligna;
2. sujeto a especificidad ≥ 0.90;
3. en empates, mayor especificidad;
4. luego mayor F1;
5. finalmente mayor threshold.

Los thresholds congelados antes de TEST fueron:

| Modelo | Threshold |
|---|---:|
| CNN | 0.17 |
| Swin-T | 0.47 |
| Multi-task | 0.05 |

El ROC-AUC no cambia al mover el threshold; sensibilidad, especificidad, F1 y accuracy sí cambian.

## 9. Swin-T: Transfer Learning y fine-tuning

Swin-T fue seleccionado como representante Transformer por su estructura jerárquica, atención por ventanas, compatibilidad con entrada rectangular 224×320 y disponibilidad de pesos `ImageNet1K V1`.

El entrenamiento tuvo dos fases:

- **Fase A:** entrenamiento de la cabeza durante 3 épocas, con el backbone congelado y `lr=1e-3`.
- **Fase B:** fine-tuning de toda la red, con `lr=1e-5` para el backbone y `lr=1e-4` para la cabeza, `weight_decay=1e-4`, máximo 20 épocas y early stopping.

La transferencia parte de representaciones visuales aprendidas en ImageNet y luego adapta los pesos al dominio BreaKHis.

In [ ]:
swin_history = pd.DataFrame([{'phase': 'A_head_warmup', 'epoch': 1, 'train_loss': 0.4299442772595608, 'val_loss': 0.350611799050376, 'roc_auc': 0.9383929947237264, 'lr_backbone': 0.0, 'lr_head': 0.001}, {'phase': 'A_head_warmup', 'epoch': 2, 'train_loss': 0.3257417418698053, 'val_loss': 0.3336228841975632, 'roc_auc': 0.9415677918820836, 'lr_backbone': 0.0, 'lr_head': 0.001}, {'phase': 'A_head_warmup', 'epoch': 3, 'train_loss': 0.2912507790986684, 'val_loss': 0.3088786912303231, 'roc_auc': 0.9518041880755148, 'lr_backbone': 0.0, 'lr_head': 0.001}, {'phase': 'B_full_finetune', 'epoch': 1, 'train_loss': 0.1644845661032419, 'val_loss': 0.6157335741679646, 'roc_auc': 0.9515703954814267, 'lr_backbone': 1e-05, 'lr_head': 0.0001}, {'phase': 'B_full_finetune', 'epoch': 2, 'train_loss': 0.0738349261302623, 'val_loss': 0.8064306235815624, 'roc_auc': 0.957521479694582, 'lr_backbone': 1e-05, 'lr_head': 0.0001}, {'phase': 'B_full_finetune', 'epoch': 3, 'train_loss': 0.0439967778229653, 'val_loss': 0.6734230657110973, 'roc_auc': 0.9641765984240254, 'lr_backbone': 1e-05, 'lr_head': 0.0001}, {'phase': 'B_full_finetune', 'epoch': 4, 'train_loss': 0.0288803032697273, 'val_loss': 1.0847146193207464, 'roc_auc': 0.9528403143447696, 'lr_backbone': 1e-05, 'lr_head': 0.0001}, {'phase': 'B_full_finetune', 'epoch': 5, 'train_loss': 0.0143042901512213, 'val_loss': 1.485149221118196, 'roc_auc': 0.9440571516623184, 'lr_backbone': 1e-05, 'lr_head': 0.0001}, {'phase': 'B_full_finetune', 'epoch': 6, 'train_loss': 0.0100789583254469, 'val_loss': 1.4600366135163896, 'roc_auc': 0.952051264339722, 'lr_backbone': 1e-05, 'lr_head': 0.0001}, {'phase': 'B_full_finetune', 'epoch': 7, 'train_loss': 0.0073821359528038, 'val_loss': 1.6253381631442907, 'roc_auc': 0.9405183819427104, 'lr_backbone': 1e-05, 'lr_head': 0.0001}, {'phase': 'B_full_finetune', 'epoch': 8, 'train_loss': 0.0171104457483867, 'val_loss': 1.5524389657424738, 'roc_auc': 0.9468786031955198, 'lr_backbone': 1e-05, 'lr_head': 0.0001}])
swin_history["epoch_global"] = np.arange(1, len(swin_history) + 1)

fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(swin_history["epoch_global"], swin_history["train_loss"], label="Train loss")
ax.plot(swin_history["epoch_global"], swin_history["val_loss"], label="Validation loss")
ax.axvline(3.5, linestyle="--", linewidth=1)
ax.set_xlabel("Época")
ax.set_ylabel("Loss")
ax.set_title("Swin-T · seed 20261007")
ax.legend()
plt.tight_layout()
fig.savefig(FIG_DIR / "swin_training_loss_historica.png", dpi=180, bbox_inches="tight")
plt.show()

fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(swin_history["epoch_global"], swin_history["roc_auc"])
ax.axvline(3.5, linestyle="--", linewidth=1)
ax.set_xlabel("Época")
ax.set_ylabel("ROC-AUC")
ax.set_title("Swin-T · ROC-AUC de VALIDATION")
plt.tight_layout()
fig.savefig(FIG_DIR / "swin_validation_auc_historica.png", dpi=180, bbox_inches="tight")
plt.show()

El procedimiento de fine-tuning puede resumirse así:

```python
for parameter in model.features.parameters():
    parameter.requires_grad = False

train_head(model, epochs=3, lr=1e-3)

for parameter in model.parameters():
    parameter.requires_grad = True

optimizer = AdamW([
    {"params": backbone_parameters, "lr": 1e-5},
    {"params": head_parameters, "lr": 1e-4}
], weight_decay=1e-4)

fine_tune(model, optimizer, early_stopping=True)
```

En la reproducción final no se descargan pesos de ImageNet ni se repite el fine-tuning; los checkpoints contienen los pesos finales completos.

## 10. CNN Multi-task

La tercera familia conserva el backbone CNN Shallow y añade dos cabezas:

- `binary_head: 128 → 1`
- `subtype_head: 128 → 8`

La tarea binaria sigue siendo la tarea principal. El subtipo funciona como supervisión auxiliar.

\[
L = L_{binaria} + \lambda L_{subtipo}
\]

Se evaluaron valores de λ en un piloto y posteriormente se repitieron `0.00`, `0.10` y `1.00` con tres seeds. La selección oficial fue `λ=0.10`.

In [ ]:
lambda_selection = pd.DataFrame([
    {"Lambda": 0.10, "AUC_medio": 0.967924, "SD": 0.007649},
    {"Lambda": 1.00, "AUC_medio": 0.966038, "SD": 0.004572},
    {"Lambda": 0.00, "AUC_medio": 0.965865, "SD": 0.008542}
]).sort_values("Lambda")

display(lambda_selection)

fig, ax = plt.subplots(figsize=(6, 4))
lambda_selection.plot(x="Lambda", y="AUC_medio", kind="bar", yerr="SD", legend=False, ax=ax)
ax.set_ylim(0.94, 0.99)
ax.set_ylabel("ROC-AUC medio")
ax.set_xlabel("λ")
ax.set_title("Selección Multi-task · 3 seeds")
plt.tight_layout()
fig.savefig(FIG_DIR / "multitask_lambda_selection.png", dpi=180, bbox_inches="tight")
plt.show()

La pérdida utilizada fue:

```python
binary_logits, subtype_logits = model(images)
loss_binary = bce(binary_logits, y_binary)
loss_subtype = cross_entropy(subtype_logits, y_subtype)
loss = loss_binary + 0.10 * loss_subtype
```

La CNN Multi-task se entrenó desde cero; no se inicializó desde el checkpoint de la CNN binaria.

# Reproducción de los predictores finales

A partir de este punto se ejecuta la parte reproducible del notebook. No hay entrenamiento, búsqueda de hiperparámetros ni selección de threshold.

El flujo es:

`dataset + manifest + 9 checkpoints → TEST → ensemble → métricas → figuras`.

In [ ]:
test_meta = manifest[manifest["split_clean"].eq("test")].copy().reset_index(drop=True)

if len(test_meta) != 1346:
    raise RuntimeError("TEST debe contener 1.346 imágenes.")
if test_meta[patient_col].astype(str).nunique() != 14:
    raise RuntimeError("TEST debe contener 14 pacientes.")
if int((test_meta["y_true"] == 0).sum()) != 411:
    raise RuntimeError("TEST debe contener 411 imágenes benignas.")
if int((test_meta["y_true"] == 1).sum()) != 935:
    raise RuntimeError("TEST debe contener 935 imágenes malignas.")
if not test_meta["sample_key"].is_unique:
    raise RuntimeError("sample_key debe ser único en TEST.")

required = set(test_meta["sample_key"])
available = {p.name for p in TEST_DIR.iterdir() if p.is_file()}
missing = required - available

if missing:
    found = set()
    with tarfile.open(DATASET_TGZ, "r:gz") as tf:
        for member in tf:
            if not member.isfile():
                continue
            basename = Path(member.name).name
            if basename not in missing:
                continue
            src = tf.extractfile(member)
            if src is None:
                continue
            with open(TEST_DIR / basename, "wb") as dst:
                shutil.copyfileobj(src, dst)
            found.add(basename)

remaining = [name for name in required if not (TEST_DIR / name).exists()]
if remaining:
    raise RuntimeError(f"Faltan {len(remaining)} imágenes de TEST.")

print("TEST materializado:", len(required), "imágenes")

In [ ]:
class TestDataset(Dataset):
    def __init__(self, frame):
        self.frame = frame.reset_index(drop=True)

    def __len__(self):
        return len(self.frame)

    def __getitem__(self, idx):
        row = self.frame.iloc[idx]
        with Image.open(TEST_DIR / row["sample_key"]) as img:
            cnn_image = resize_pad_normalize(img, CNN_MEAN, CNN_STD)
            swin_image = resize_pad_normalize(img, IMAGENET_MEAN, IMAGENET_STD)

        return {
            "cnn_image": cnn_image,
            "swin_image": swin_image,
            "sample_key": row["sample_key"],
            "y_true": int(row["y_true"])
        }

class ConvBNReLU(nn.Sequential):
    def __init__(self, in_channels, out_channels, kernel_size=3, stride=1):
        padding = kernel_size // 2
        super().__init__(
            nn.Conv2d(in_channels, out_channels, kernel_size, stride, padding, bias=False),
            nn.BatchNorm2d(out_channels),
            nn.ReLU(inplace=True)
        )

class CNNShallow(nn.Module):
    def __init__(self, dropout=0.30):
        super().__init__()
        self.features = nn.Sequential(
            ConvBNReLU(3, 32),
            nn.MaxPool2d(2),
            ConvBNReLU(32, 64),
            nn.MaxPool2d(2),
            ConvBNReLU(64, 128),
            nn.MaxPool2d(2)
        )
        self.gap = nn.AdaptiveAvgPool2d((1, 1))
        self.dropout = nn.Dropout(dropout)
        self.classifier = nn.Linear(128, 1)

    def forward(self, x):
        x = self.features(x)
        x = self.gap(x)
        x = torch.flatten(x, 1)
        x = self.dropout(x)
        return self.classifier(x).squeeze(1)

class CNNShallowMultiTask(nn.Module):
    def __init__(self, dropout=0.30):
        super().__init__()
        self.features = nn.Sequential(
            ConvBNReLU(3, 32),
            nn.MaxPool2d(2),
            ConvBNReLU(32, 64),
            nn.MaxPool2d(2),
            ConvBNReLU(64, 128),
            nn.MaxPool2d(2)
        )
        self.gap = nn.AdaptiveAvgPool2d((1, 1))
        self.dropout = nn.Dropout(dropout)
        self.binary_head = nn.Linear(128, 1)
        self.subtype_head = nn.Linear(128, 8)

    def forward(self, x):
        x = self.features(x)
        x = self.gap(x)
        x = torch.flatten(x, 1)
        x = self.dropout(x)
        return self.binary_head(x).squeeze(1), self.subtype_head(x)

def build_swin():
    model = swin_t(weights=None)
    model.head = nn.Linear(model.head.in_features, 1)
    return model

def build_model(family):
    if family == "CNN":
        return CNNShallow()
    if family == "Swin-T":
        return build_swin()
    if family == "Multi-task":
        return CNNShallowMultiTask()
    raise ValueError(family)

parameter_counts = {
    family: sum(p.numel() for p in build_model(family).parameters())
    for family in ["CNN", "Swin-T", "Multi-task"]
}

expected_params = {"CNN": 93601, "Swin-T": 27520123, "Multi-task": 94633}
if parameter_counts != expected_params:
    raise RuntimeError(parameter_counts)

display(pd.Series(parameter_counts, name="Parámetros").to_frame())

In [ ]:
def torch_load(path):
    try:
        return torch.load(path, map_location="cpu", weights_only=False)
    except TypeError:
        return torch.load(path, map_location="cpu")

def load_model(family, seed, gradients=False):
    filename, _ = CHECKPOINTS[family][seed]
    obj = torch_load(CHECKPOINT_DIR / filename)

    if obj.get("manifest_sha256") != MANIFEST_SHA:
        raise RuntimeError(f"Manifest distinto en {filename}")

    if family == "Multi-task":
        observed_lambda = obj.get("lambda_aux")
        if observed_lambda is None or abs(float(observed_lambda) - MULTITASK_LAMBDA) > 1e-12:
            raise RuntimeError(f"Lambda incorrecto en {filename}")

    model = build_model(family)
    model.load_state_dict(obj["state_dict"], strict=True)
    model.to(DEVICE)
    model.eval()

    if not gradients:
        for p in model.parameters():
            p.requires_grad_(False)

    return model

dataset = TestDataset(test_meta)
loader = DataLoader(
    dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=2,
    pin_memory=torch.cuda.is_available(),
    persistent_workers=True,
    drop_last=False
)

@torch.inference_mode()
def infer_family(family):
    input_key = "swin_image" if family == "Swin-T" else "cnn_image"
    models = [load_model(family, seed) for seed in SEEDS]

    keys = []
    labels = []
    seed_probs = {seed: [] for seed in SEEDS}

    for batch in loader:
        x = batch[input_key].to(DEVICE, non_blocking=True)

        for seed, model in zip(SEEDS, models):
            output = model(x)
            logits = output[0] if family == "Multi-task" else output
            if family == "Swin-T":
                logits = logits.squeeze(1)
            probs = torch.sigmoid(logits).cpu().numpy()
            seed_probs[seed].extend(probs.tolist())

        keys.extend(list(batch["sample_key"]))
        labels.extend(np.asarray(batch["y_true"], dtype=int).tolist())

    matrix = np.column_stack([np.asarray(seed_probs[seed], dtype=float) for seed in SEEDS])

    frame = pd.DataFrame({
        "sample_key": keys,
        "y_true": labels,
        f"{family}_seed_20261007": matrix[:, 0],
        f"{family}_seed_20261008": matrix[:, 1],
        f"{family}_seed_20261009": matrix[:, 2],
        f"{family}_probability": matrix.mean(axis=1)
    })

    del models
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

    return frame

predictions = test_meta[["sample_key", patient_col, "subtype_clean", "y_true"]].copy()
predictions = predictions.rename(columns={patient_col: "patient_id", "subtype_clean": "subtype"})

for family in ["CNN", "Swin-T", "Multi-task"]:
    frame = infer_family(family)
    predictions = predictions.merge(
        frame.drop(columns="y_true"),
        on="sample_key",
        how="inner",
        validate="one_to_one"
    )
    print(family, "completado")

predictions.to_csv(RESULTS_DIR / "test_predictions.csv", index=False)

## 11. Resultados finales sobre el mismo TEST

Las tres familias se evalúan sobre **1.346 imágenes de 14 pacientes**. Ninguna decisión de arquitectura, hiperparámetro, λ o threshold se modifica después de observar TEST.

In [ ]:
def binary_metrics(y_true, probability, threshold):
    y = np.asarray(y_true, dtype=int)
    p = np.asarray(probability, dtype=float)
    pred = (p >= threshold).astype(int)

    tn, fp, fn, tp = confusion_matrix(y, pred, labels=[0, 1]).ravel()
    sensitivity = tp / (tp + fn)
    specificity = tn / (tn + fp)

    return {
        "ROC-AUC": roc_auc_score(y, p),
        "Sensibilidad": sensitivity,
        "Especificidad": specificity,
        "Precision": precision_score(y, pred, zero_division=0),
        "F1": f1_score(y, pred, zero_division=0),
        "Accuracy": accuracy_score(y, pred),
        "Balanced Accuracy": (sensitivity + specificity) / 2,
        "TN": int(tn),
        "FP": int(fp),
        "FN": int(fn),
        "TP": int(tp)
    }

rows = []

for family in ["CNN", "Swin-T", "Multi-task"]:
    metrics = binary_metrics(
        predictions["y_true"],
        predictions[f"{family}_probability"],
        THRESHOLDS[family]
    )

    expected = EXPECTED[family]

    if abs(metrics["ROC-AUC"] - expected["auc"]) > 5e-6:
        raise RuntimeError(f"ROC-AUC no reproducido para {family}")

    for key in ["tn", "fp", "fn", "tp"]:
        if metrics[key.upper()] != expected[key]:
            raise RuntimeError(f"Matriz de confusión no reproducida para {family}")

    rows.append({"Modelo": family, "Threshold": THRESHOLDS[family], **metrics})

metrics_df = pd.DataFrame(rows)
metrics_df.to_csv(RESULTS_DIR / "test_metrics.csv", index=False)

display(
    metrics_df[
        ["Modelo", "ROC-AUC", "Sensibilidad", "Especificidad", "Precision", "F1",
         "Accuracy", "Balanced Accuracy", "FN", "FP"]
    ].style.format({
        "ROC-AUC": "{:.4f}",
        "Sensibilidad": "{:.2%}",
        "Especificidad": "{:.2%}",
        "Precision": "{:.2%}",
        "F1": "{:.2%}",
        "Accuracy": "{:.2%}",
        "Balanced Accuracy": "{:.2%}"
    })
)

In [ ]:
fig, ax = plt.subplots(figsize=(7, 5))

for family in ["CNN", "Swin-T", "Multi-task"]:
    fpr, tpr, _ = roc_curve(predictions["y_true"], predictions[f"{family}_probability"])
    auc = metrics_df.loc[metrics_df["Modelo"] == family, "ROC-AUC"].iloc[0]
    ax.plot(fpr, tpr, label=f"{family} (AUC={auc:.3f})")

ax.plot([0, 1], [0, 1], linestyle="--", linewidth=1)
ax.set_xlabel("False Positive Rate")
ax.set_ylabel("True Positive Rate")
ax.set_title("Curvas ROC finales en TEST")
ax.legend()
plt.tight_layout()
fig.savefig(FIG_DIR / "curvas_roc_test.png", dpi=180, bbox_inches="tight")
plt.show()

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(12, 4))

for ax, family in zip(axes, ["CNN", "Swin-T", "Multi-task"]):
    row = metrics_df.loc[metrics_df["Modelo"] == family].iloc[0]
    cm = np.array([[row["TN"], row["FP"]], [row["FN"], row["TP"]]])

    ax.imshow(cm)
    for i in range(2):
        for j in range(2):
            ax.text(j, i, int(cm[i, j]), ha="center", va="center")

    ax.set_xticks([0, 1], ["Benigno", "Maligno"])
    ax.set_yticks([0, 1], ["Benigno", "Maligno"])
    ax.set_xlabel("Predicción")
    ax.set_ylabel("Real")
    ax.set_title(f"{family} · τ={THRESHOLDS[family]}")

plt.tight_layout()
fig.savefig(FIG_DIR / "matrices_confusion_test.png", dpi=180, bbox_inches="tight")
plt.show()

In [ ]:
comparison = metrics_df.set_index("Modelo")[["ROC-AUC", "Sensibilidad", "Especificidad", "F1", "Balanced Accuracy"]]

fig, ax = plt.subplots(figsize=(9, 5))
comparison.plot(kind="bar", ax=ax)
ax.set_ylabel("Métrica")
ax.set_xlabel("")
ax.set_ylim(0, 1)
ax.set_title("Comparación final en TEST")
ax.tick_params(axis="x", rotation=0)
plt.tight_layout()
fig.savefig(FIG_DIR / "comparacion_metricas_test.png", dpi=180, bbox_inches="tight")
plt.show()

### Interpretación

- **Swin-T** lidera ROC-AUC, especificidad, precision, F1, accuracy y balanced accuracy.
- **Multi-task** alcanza la mayor sensibilidad y reduce los falsos negativos a 12.
- **CNN Shallow** mantiene una sensibilidad alta con una arquitectura extremadamente pequeña.

No se definió antes de TEST una función escalar que combinara todos los objetivos. Por ello, la lectura final es un trade-off: Swin-T para equilibrio global, Multi-task para sensibilidad y CNN para parsimonia.

## 12. Costo computacional

La comparación de inferencia se realizó bajo las mismas condiciones en una Tesla T4, con batch 32 y tres seeds por ensemble.

In [ ]:
benchmark = pd.DataFrame([
    {"Modelo": "CNN", "Parametros": 93601, "Forward_ms_img": 2.508, "img_s": 398.75, "End_to_end_ms_img": 17.836},
    {"Modelo": "Swin-T", "Parametros": 27520123, "Forward_ms_img": 18.187, "img_s": 54.98, "End_to_end_ms_img": 21.891},
    {"Modelo": "Multi-task", "Parametros": 94633, "Forward_ms_img": 2.539, "img_s": 393.82, "End_to_end_ms_img": 17.888}
])

display(benchmark)

fig, ax = plt.subplots(figsize=(7, 4))
benchmark.set_index("Modelo")["Parametros"].plot(kind="bar", logy=True, ax=ax)
ax.set_ylabel("Parámetros, escala logarítmica")
ax.set_xlabel("")
ax.set_title("Complejidad estructural")
ax.tick_params(axis="x", rotation=0)
plt.tight_layout()
fig.savefig(FIG_DIR / "comparacion_parametros.png", dpi=180, bbox_inches="tight")
plt.show()

Swin-T tiene aproximadamente 291 veces más parámetros que Multi-task y el model-forward del ensemble es unas 7,2 veces más lento. Multi-task añade solo 1.032 parámetros a la CNN Shallow.

## 13. Grad-CAM

Grad-CAM se interpreta como **atribución de decisión**: indica qué regiones tuvieron mayor influencia relativa sobre una salida del modelo. No debe interpretarse como segmentación, localización tumoral validada ni evidencia clínica.

La siguiente celda genera dos ejemplos reproducibles con la CNN Multi-task: un benigno correctamente clasificado y un maligno correctamente clasificado. Para cada caso se promedian los mapas de las tres seeds del ensemble.

In [ ]:
def gradcam_binary(model, x):
    activations = {}
    gradients = {}

    target = model.features[4]

    def forward_hook(module, inp, out):
        activations["value"] = out

    def backward_hook(module, grad_input, grad_output):
        gradients["value"] = grad_output[0]

    h1 = target.register_forward_hook(forward_hook)
    h2 = target.register_full_backward_hook(backward_hook)

    model.zero_grad(set_to_none=True)
    x = x.clone().detach().to(DEVICE).requires_grad_(True)

    binary_logit, _ = model(x)
    binary_logit.squeeze().backward()

    a = activations["value"]
    g = gradients["value"]
    weights = g.mean(dim=(2, 3), keepdim=True)
    cam = torch.relu((weights * a).sum(dim=1, keepdim=True))
    cam = F.interpolate(cam, size=(TARGET_H, TARGET_W), mode="bilinear", align_corners=False)
    cam = cam[0, 0].detach().cpu().numpy()

    if cam.max() > cam.min():
        cam = (cam - cam.min()) / (cam.max() - cam.min())
    else:
        cam = np.zeros_like(cam)

    h1.remove()
    h2.remove()
    return cam

mt_models = [load_model("Multi-task", seed, gradients=True) for seed in SEEDS]

mt_prob = predictions["Multi-task_probability"].to_numpy()
y = predictions["y_true"].to_numpy()
pred = (mt_prob >= THRESHOLDS["Multi-task"]).astype(int)

tn_idx = np.where((y == 0) & (pred == 0))[0]
tp_idx = np.where((y == 1) & (pred == 1))[0]

selected = {
    "Benigno correctamente clasificado": int(tn_idx[np.argmin(mt_prob[tn_idx])]),
    "Maligno correctamente clasificado": int(tp_idx[np.argmax(mt_prob[tp_idx])])
}

fig, axes = plt.subplots(2, 2, figsize=(10, 7))

for row_idx, (title, idx) in enumerate(selected.items()):
    sample_key = predictions.iloc[idx]["sample_key"]

    with Image.open(TEST_DIR / sample_key) as img:
        display_img = resize_pad_display(img)
        x = resize_pad_normalize(img, CNN_MEAN, CNN_STD).unsqueeze(0)

    cams = [gradcam_binary(model, x) for model in mt_models]
    ensemble_cam = np.mean(cams, axis=0)

    axes[row_idx, 0].imshow(display_img)
    axes[row_idx, 0].set_title(title)
    axes[row_idx, 0].axis("off")

    axes[row_idx, 1].imshow(display_img)
    axes[row_idx, 1].imshow(ensemble_cam, alpha=0.45, cmap="inferno")
    axes[row_idx, 1].set_title("Grad-CAM ensemble")
    axes[row_idx, 1].axis("off")

plt.tight_layout()
fig.savefig(FIG_DIR / "gradcam_multitask_ejemplos.png", dpi=180, bbox_inches="tight")
plt.show()

del mt_models
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()

## 14. Limitaciones

- Se trabajó con un único dataset: BreaKHis.
- TEST contiene 14 pacientes, de los cuales 4 son benignos y 10 malignos.
- Swin-T fue el único Transformer entrenado; el estudio compara familias y no pretende establecer superioridad universal sobre ViT o DeiT.
- La cabeza auxiliar de subtipo se diseñó como señal de entrenamiento y no como clasificador clínico robusto de ocho subtipos.
- Grad-CAM aporta atribución visual, no localización histopatológica validada ni explicación causal.
- La validez externa requiere evaluación en otros centros, otros equipos y otras distribuciones de datos.

## 15. Alcance, propuesta de valor e impacto

BH-CNN es un pipeline académico reproducible que integra auditoría de datos, split por paciente, CNN desde cero, Transfer Learning, aprendizaje Multi-task, ensembles, selección de threshold, evaluación final y explicabilidad visual.

Su propuesta de valor está en comparar estrategias bajo un mismo protocolo y un mismo TEST, manteniendo trazabilidad de las decisiones y evitando cambios retrospectivos.

Usuarios potenciales en una etapa posterior de investigación y validación incluyen patólogos, investigadores en patología computacional, científicos de datos biomédicos, hospitales, laboratorios, centros oncológicos y universidades.

El proyecto se relaciona de forma indirecta con:

- **ODS 3: Salud y bienestar**, por investigar métodos de inteligencia artificial aplicados a imágenes relacionadas con cáncer de mama.
- **ODS 9: Industria, innovación e infraestructura**, por desarrollar y evaluar soluciones reproducibles de Deep Learning considerando desempeño, costo y trazabilidad.

En su estado actual, BH-CNN no debe presentarse como herramienta diagnóstica clínica validada.

## 16. Conclusiones

Tres estrategias distintas alcanzaron un desempeño alto sobre el mismo split por paciente, pero sus perfiles de error no son equivalentes.

- Swin-T logró el mejor equilibrio global: ROC-AUC 0.983156 y balanced accuracy 0.928704.
- Multi-task alcanzó la mayor sensibilidad: 0.987166 y 12 falsos negativos.
- CNN Shallow mantuvo la máxima parsimonia: 93.601 parámetros.

La principal conclusión metodológica es que el resultado depende tanto de la arquitectura como del protocolo experimental. Auditar duplicados, dividir por paciente, congelar decisiones antes de TEST, repetir seeds y evitar cambios post-hoc permitió comparar los modelos con mayor confianza.

In [ ]:
summary = {
    "training_executed_in_this_notebook": False,
    "retuning_executed_in_this_notebook": False,
    "test_images": int(len(test_meta)),
    "test_patients": int(test_meta[patient_col].astype(str).nunique()),
    "seeds": SEEDS,
    "thresholds": THRESHOLDS,
    "multitask_lambda": MULTITASK_LAMBDA,
    "manifest_sha256": sha256_file(MANIFEST_PATH),
    "metrics": metrics_df.to_dict("records")
}

with open(RESULTS_DIR / "reproducibility_summary.json", "w", encoding="utf-8") as f:
    json.dump(summary, f, indent=2, ensure_ascii=False)

archive_base = str(PREFERRED / "BH_CNN_MEDIUM_RESULTS")
archive_path = shutil.make_archive(archive_base, "zip", root_dir=RESULTS_DIR)

print("Resultados guardados en:", RESULTS_DIR)
print("ZIP de resultados:", archive_path)
print("No se realizó entrenamiento ni retuning.")

## Referencias

1. Spanhol, F. A., Oliveira, L. S., Petitjean, C., & Heutte, L. (2016). *A Dataset for Breast Cancer Histopathological Image Classification*. IEEE Transactions on Biomedical Engineering, 63(7), 1455–1462.
2. Liu, Z. et al. (2021). *Swin Transformer: Hierarchical Vision Transformer Using Shifted Windows*. ICCV.
3. Selvaraju, R. R. et al. (2017). *Grad-CAM: Visual Explanations From Deep Networks via Gradient-Based Localization*. ICCV.
4. Caruana, R. (1997). *Multitask Learning*. Machine Learning, 28, 41–75.